# 🔒 Pipeline Completo de Análisis de Seguridad

**Curso:** Ciberseguridad (ICC610) - 2026  
**Objetivo:** Ejecutar el pipeline completo de análisis de seguridad y realizar el análisis cuantitativo, todo desde un solo notebook.

## ¿Qué hace este notebook?

| Fase | Descripción | Herramienta |
|------|-------------|-------------|
| **1** | Configuración e imports | Python |
| **2** | Clonar repositorios | Git |
| **3** | Generar SBOM (lista de dependencias) | Syft |
| **4** | Escanear vulnerabilidades en dependencias | Grype |
| **5** | Análisis estático del código fuente | CodeQL |
| **6** | Generar reporte consolidado | Python |
| **7** | Análisis cuantitativo de dependencias | Pandas |
| **8** | Análisis cuantitativo de vulnerabilidades | Pandas |
| **9** | Análisis cuantitativo de CodeQL | Pandas |
| **10** | Resumen ejecutivo | Python |
| **11** | Exportar a CSV | Pandas |

> ⚠️ **Nota:** Ejecuta las celdas en orden. La primera ejecución puede tardar 10-20 minutos (clonado + CodeQL).

---

## 1. ⚙️ Configuración e Imports

In [1]:
import json
import subprocess
import sys
import os
import shutil
import pandas as pd
from pathlib import Path
from collections import Counter
from datetime import datetime

# Configurar directorio de trabajo
PROJECT_ROOT = Path("/workspaces/sbom-vuln-analysis")
REPOS_DIR = PROJECT_ROOT / "data" / "repos"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
CONFIG_FILE = PROJECT_ROOT / "data" / "config.json"

# Asegurar que existen los directorios
REPOS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Agregar scripts al path
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

print("═" * 60)
print("  🔒 PIPELINE COMPLETO DE ANÁLISIS DE SEGURIDAD")
print("═" * 60)
print(f"  📂 Proyecto:   {PROJECT_ROOT}")
print(f"  📂 Repos:      {REPOS_DIR}")
print(f"  📂 Resultados: {RESULTS_DIR}")
print(f"  📄 Config:     {CONFIG_FILE}")
print(f"  🕐 Fecha:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("═" * 60)

# Verificar herramientas disponibles
print("\n🔧 Verificando herramientas instaladas...")
tools = {"syft": "syft version", "grype": "grype version", "codeql": "codeql version"}
tools_ok = True
for tool, cmd in tools.items():
    result = subprocess.run(cmd.split(), capture_output=True, text=True)
    version = result.stdout.strip().split("\n")[0] if result.returncode == 0 else "❌ NO ENCONTRADO"
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {tool:10} → {version}")
    if result.returncode != 0:
        tools_ok = False

if tools_ok:
    print("\n✅ Todas las herramientas están disponibles.")
else:
    print("\n⚠️  Algunas herramientas no están disponibles. Algunos pasos podrían fallar.")

════════════════════════════════════════════════════════════
  🔒 PIPELINE COMPLETO DE ANÁLISIS DE SEGURIDAD
════════════════════════════════════════════════════════════
  📂 Proyecto:   /workspaces/sbom-vuln-analysis
  📂 Repos:      /workspaces/sbom-vuln-analysis/data/repos
  📂 Resultados: /workspaces/sbom-vuln-analysis/data/results
  📄 Config:     /workspaces/sbom-vuln-analysis/data/config.json
  🕐 Fecha:      2026-04-14 02:08:10
════════════════════════════════════════════════════════════

🔧 Verificando herramientas instaladas...
  ✅ syft       → Application:   syft
  ✅ grype      → Application:         grype
  ✅ codeql     → CodeQL command-line toolchain release 2.25.1.

✅ Todas las herramientas están disponibles.


### 1.1 Configurar repositorios a analizar

Modifica la lista `REPOSITORIES` para cambiar qué repositorios analizar.  
El `config.json` se actualizará automáticamente.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURACIÓN: Repositorios a analizar                     ║
# ║  Este bloque lee la configuración actual. Para cambiarla,   ║
# ║  edita directamente el archivo data/config.json             ║
# ╚══════════════════════════════════════════════════════════════╝

if CONFIG_FILE.exists():
    with open(CONFIG_FILE, "r") as f:
        config = json.load(f)
    print(f"✅ Cargando configuración existente desde {CONFIG_FILE}")
else:
    print(f"⚠️ No se encontró {CONFIG_FILE}. Creando configuración por defecto...")
    config = {
        "repos_dir": "data/repos",
        "output_dir": "data/results",
        "repositories": ["https://github.com/pallets/flask"],
        "organizations": [],
        "clone_options": {
            "max_inactive_days": 30,
            "skip_archived": True,
            "skip_forks": True,
            "max_repos": 50
        },
        "tools": {
            "syft": {"enabled": True, "output_format": "json"},
            "grype": {"enabled": True, "output_format": "json", "update_db": True},
            "codeql": {
                "enabled": True,
                "output_format": "json",
                "supported_languages": ["python", "javascript", "java", "cpp", "csharp"]
            }
        },
        "concurrency": {
            "max_workers": 4,
            "enabled": True
        }
    }
    with open(CONFIG_FILE, "w") as f:
        json.dump(config, f, indent=4)

REPOSITORIES = config.get("repositories", [])
ORGANIZATIONS = config.get("organizations", [])

print("\n�� Configuración actual:")
print(f"  Repositorios individuales: {len(REPOSITORIES)}")
for r in REPOSITORIES:
    print(f"    → {r}")
if ORGANIZATIONS:
    print(f"  Organizaciones: {len(ORGANIZATIONS)}")
    for o in ORGANIZATIONS:
        print(f"    → {o}")


---

## 2. 📥 Clonar Repositorios

Descarga los repositorios configurados en `data/repos/`.

In [ ]:
print("═" * 60)
print("  [1/5] 📥 CLONANDO REPOSITORIOS")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "clone"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    # Listar repos clonados
    repos = [d.name for d in REPOS_DIR.iterdir() if d.is_dir()]
    print(f"\n📂 Repositorios disponibles en data/repos/: {repos}")

---

## 3. 📦 Generar SBOMs (Syft)

Genera la lista de componentes y dependencias de cada repositorio usando **Syft**.

In [ ]:
print("═" * 60)
print("  [2/5] 📦 GENERANDO SBOMs (SYFT)")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "sbom"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    sbom_files = sorted(RESULTS_DIR.glob("*-sbom.json"))
    print(f"\n✅ {len(sbom_files)} SBOM(s) generados")
    for f in sbom_files:
        size = f.stat().st_size / 1024
        print(f"   📄 {f.name} ({size:.1f} KB)")

---

## 4. 🔓 Escanear Vulnerabilidades (Grype)

Busca CVEs conocidos en las dependencias detectadas usando **Grype**.

> ℹ️ La primera ejecución descarga la base de datos de vulnerabilidades (~2 min).

In [ ]:
print("═" * 60)
print("  [3/5] 🔓 ESCANEANDO VULNERABILIDADES (GRYPE)")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "grype"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
    timeout=600
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    grype_files = sorted(RESULTS_DIR.glob("*-grype.json"))
    print(f"\n✅ {len(grype_files)} escaneo(s) completados")
    for f in grype_files:
        with open(f) as fh:
            data = json.load(fh)
        n_vulns = len(data.get("matches", []))
        print(f"   🔓 {f.stem}: {n_vulns} vulnerabilidades")

---

## 5. 🔍 Análisis Estático de Código (CodeQL)

Encuentra vulnerabilidades en el código fuente (SQL injection, XSS, etc.) usando **CodeQL**.

> ⏳ **Este paso puede tardar 5-20 minutos** por repositorio. Si deseas saltarlo, comenta la celda y continúa.

In [ ]:
print("═" * 60)
print("  [4/5] 🔍 ANÁLISIS ESTÁTICO (CODEQL)")
print("═" * 60)
print("⏳ Este paso puede tardar varios minutos...\n")

result = subprocess.run(
    ["uv", "run", "python", "main.py", "codeql"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
    timeout=1800  # 30 min max
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
else:
    codeql_files = sorted(RESULTS_DIR.glob("*-codeql.json"))
    print(f"\n✅ {len(codeql_files)} análisis completados")
    for f in codeql_files:
        with open(f) as fh:
            data = json.load(fh)
        n_findings = data.get("total", len(data.get("findings", [])))
        print(f"   🔍 {f.stem}: {n_findings} hallazgos")

---

## 6. 📊 Generar Reporte Consolidado

In [ ]:
print("═" * 60)
print("  [5/5] 📊 GENERANDO REPORTE CONSOLIDADO")
print("═" * 60)

result = subprocess.run(
    ["uv", "run", "python", "main.py", "report"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Errores:")
    print(result.stderr)
else:
    report_file = RESULTS_DIR / "consolidated-report.json"
    if report_file.exists():
        with open(report_file) as f:
            report = json.load(f)
        print("\n📋 Contenido del reporte:")
        print(json.dumps(report, indent=2))

print("\n" + "═" * 60)
print("  ✅ PIPELINE COMPLETO EJECUTADO")
print("═" * 60)

# Resumen de archivos generados
print("\n📂 Archivos generados en data/results/:")
for f in sorted(RESULTS_DIR.iterdir()):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"   {'📄' if size < 100 else '📦'} {f.name:40} ({size:.1f} KB)")

---

# 📊 ANÁLISIS CUANTITATIVO

A partir de aquí se analizan cuantitativamente los resultados generados por el pipeline.

---

## 7. 📦 Análisis de Dependencias (SBOM)

Analizamos los componentes de software detectados por Syft en cada repositorio.

In [2]:
# Cargar todos los SBOMs
sbom_files = sorted(RESULTS_DIR.glob("*-sbom.json"))
grype_files = sorted(RESULTS_DIR.glob("*-grype.json"))
codeql_files = sorted(RESULTS_DIR.glob("*-codeql.json"))

print("═" * 60)
print("  📂 ARCHIVOS DE RESULTADOS ENCONTRADOS")
print("═" * 60)
print(f"  📦 SBOMs:           {len(sbom_files)} archivo(s)")
print(f"  🔓 Grype (vulns):   {len(grype_files)} archivo(s)")
print(f"  🔍 CodeQL:          {len(codeql_files)} archivo(s)")
print("═" * 60)

if not sbom_files and not grype_files:
    print("\n⚠️  No se encontraron resultados. Ejecuta las celdas anteriores del pipeline.")

════════════════════════════════════════════════════════════
  📂 ARCHIVOS DE RESULTADOS ENCONTRADOS
════════════════════════════════════════════════════════════
  📦 SBOMs:           1 archivo(s)
  🔓 Grype (vulns):   1 archivo(s)
  🔍 CodeQL:          1 archivo(s)
════════════════════════════════════════════════════════════


In [3]:
# Cargar todos los SBOMs y extraer dependencias
all_dependencies = []

for sbom_file in sbom_files:
    repo_name = sbom_file.stem.replace("-sbom", "")
    
    with open(sbom_file) as f:
        data = json.load(f)
    
    artifacts = data.get("artifacts", [])
    
    for art in artifacts:
        all_dependencies.append({
            "repo": repo_name,
            "name": art.get("name", "N/A"),
            "version": art.get("version", "N/A"),
            "type": art.get("type", "N/A"),
            "language": art.get("language", "N/A"),
            "licenses": ", ".join(
                [lic.get("value", "N/A") for lic in art.get("licenses", [])]
            ) or "No especificada",
        })

df_deps = pd.DataFrame(all_dependencies)

if not df_deps.empty:
    print(f"\n📦 Total de dependencias detectadas: {len(df_deps)}")
    print(f"📁 Repositorios analizados: {df_deps['repo'].nunique()}")
    print(f"\n--- Primeras 10 dependencias ---")
    display(df_deps.head(10))
else:
    print("⚠️  No se encontraron dependencias en los SBOMs.")


📦 Total de dependencias detectadas: 121
📁 Repositorios analizados: 1

--- Primeras 10 dependencias ---


,repo,name,version,type,language,licenses
0,flask,actions/cache,v5.0.4,github-action,,No especificada
1,flask,actions/cache,v5.0.4,github-action,,No especificada
2,flask,actions/checkout,v6.0.2,github-action,,No especificada
3,flask,actions/checkout,v6.0.2,github-action,,No especificada
4,flask,actions/checkout,v6.0.2,github-action,,No especificada
5,flask,actions/checkout,v6.0.2,github-action,,No especificada
6,flask,actions/download-artifact,v8.0.1,github-action,,No especificada
7,flask,actions/setup-python,v6.2.0,github-action,,No especificada
8,flask,actions/setup-python,v6.2.0,github-action,,No especificada
9,flask,actions/setup-python,v6.2.0,github-action,,No especificada


### 7.1 Dependencias por tipo de paquete

In [4]:
if not df_deps.empty:
    print("\n📊 Distribución de dependencias por tipo de paquete:\n")
    type_counts = df_deps["type"].value_counts()
    
    max_count = type_counts.max()
    for pkg_type, count in type_counts.items():
        bar_len = int(count / max_count * 40)
        bar = "█" * bar_len
        pct = count / len(df_deps) * 100
        print(f"  {pkg_type:20} │ {bar} {count} ({pct:.1f}%)")
    
    print(f"\n  {'TOTAL':20} │ {len(df_deps)}")


📊 Distribución de dependencias por tipo de paquete:

  python               │ ████████████████████████████████████████ 104 (86.0%)
  github-action        │ ██████ 17 (14.0%)

  TOTAL                │ 121


### 7.2 Dependencias por repositorio

In [5]:
if not df_deps.empty:
    print("\n📊 Cantidad de dependencias por repositorio:\n")
    repo_counts = df_deps.groupby("repo").size().sort_values(ascending=False)
    
    max_count = repo_counts.max()
    for repo, count in repo_counts.items():
        bar_len = int(count / max_count * 40)
        bar = "█" * bar_len
        print(f"  {repo:30} │ {bar} {count}")
    
    print(f"\n  Promedio por repositorio: {len(df_deps) / df_deps['repo'].nunique():.1f}")


📊 Cantidad de dependencias por repositorio:

  flask                          │ ████████████████████████████████████████ 121

  Promedio por repositorio: 121.0


### 7.3 Licencias más comunes

In [6]:
if not df_deps.empty:
    print("\n📜 Top 10 licencias más comunes:\n")
    license_counts = df_deps["licenses"].value_counts().head(10)
    
    for lic, count in license_counts.items():
        pct = count / len(df_deps) * 100
        bar = "█" * max(1, int(pct))
        print(f"  {lic:35} │ {bar} {count} ({pct:.1f}%)")


📜 Top 10 licencias más comunes:

  No especificada                     │ ████████████████████████████████████████████████████████████████████████████████████████████████████ 121 (100.0%)


### 7.4 Tabla resumen de dependencias

In [7]:
if not df_deps.empty:
    summary_deps = df_deps.groupby("repo").agg(
        total_deps=("name", "count"),
        tipos_unicos=("type", "nunique"),
        paquetes_unicos=("name", "nunique"),
    ).reset_index()
    
    print("\n📋 Resumen de dependencias por repositorio:\n")
    display(summary_deps)


📋 Resumen de dependencias por repositorio:



,repo,total_deps,tipos_unicos,paquetes_unicos
0,flask,121,2,104


---

## 8. 🔓 Análisis de Vulnerabilidades (Grype)

Analizamos las vulnerabilidades detectadas en las dependencias de cada repositorio.

In [8]:
# Cargar todos los resultados de Grype
all_vulns = []

for grype_file in grype_files:
    repo_name = grype_file.stem.replace("-grype", "")
    
    with open(grype_file) as f:
        data = json.load(f)
    
    matches = data.get("matches", [])
    
    for match in matches:
        vuln = match.get("vulnerability", {})
        artifact = match.get("artifact", {})
        
        all_vulns.append({
            "repo": repo_name,
            "vuln_id": vuln.get("id", "N/A"),
            "severity": vuln.get("severity", "Unknown"),
            "description": vuln.get("description", "N/A")[:100],
            "package": artifact.get("name", "N/A"),
            "version": artifact.get("version", "N/A"),
            "pkg_type": artifact.get("type", "N/A"),
            "fix_state": vuln.get("fix", {}).get("state", "N/A"),
            "fix_versions": ", ".join(vuln.get("fix", {}).get("versions", [])),
            "data_source": vuln.get("dataSource", "N/A"),
        })

df_vulns = pd.DataFrame(all_vulns)

if not df_vulns.empty:
    print(f"\n🔓 Total de vulnerabilidades detectadas: {len(df_vulns)}")
    print(f"📁 Repositorios con vulnerabilidades: {df_vulns['repo'].nunique()}")
    print(f"📦 Paquetes afectados: {df_vulns['package'].nunique()}")
    print(f"\n--- Todas las vulnerabilidades ---")
    display(df_vulns[["repo", "vuln_id", "severity", "package", "version", "fix_state", "fix_versions"]])
else:
    print("⚠️  No se encontraron vulnerabilidades (o no se ejecutó Grype).")


🔓 Total de vulnerabilidades detectadas: 15
📁 Repositorios con vulnerabilidades: 1
📦 Paquetes afectados: 5

--- Todas las vulnerabilidades ---


,repo,vuln_id,severity,package,version,fix_state,fix_versions
0,flask,GHSA-2g68-c3qc-8985,High,werkzeug,2.3.3,fixed,3.0.3
1,flask,GHSA-f9vj-2wh5-fj8j,Medium,werkzeug,2.3.3,fixed,3.0.6
2,flask,GHSA-q34m-jh98-gwm2,Medium,werkzeug,2.3.3,fixed,3.0.6
3,flask,GHSA-h75v-3vvj-5mfj,Medium,jinja2,3.1.2,fixed,3.1.4
4,flask,GHSA-hrfv-mqp8-q5rw,Medium,werkzeug,2.3.3,fixed,2.3.8
5,flask,GHSA-gmj6-6f8f-6699,Medium,jinja2,3.1.2,fixed,3.1.5
6,flask,GHSA-q2x7-8rv6-6q7h,Medium,jinja2,3.1.2,fixed,3.1.5
7,flask,GHSA-cpwx-vrp4-4pq7,Medium,jinja2,3.1.2,fixed,3.1.6
8,flask,GHSA-h5c8-rqwp-cp95,Medium,jinja2,3.1.2,fixed,3.1.3
9,flask,GHSA-p423-j2cm-9vmq,Medium,cryptography,46.0.6,fixed,46.0.7


### 8.1 Distribución de vulnerabilidades por severidad

In [9]:
if not df_vulns.empty:
    print("\n🎯 Distribución por severidad:\n")
    
    severity_order = ["Critical", "High", "Medium", "Low", "Negligible", "Unknown"]
    severity_colors = {
        "Critical": "🔴",
        "High": "🟠",
        "Medium": "🟡",
        "Low": "🟢",
        "Negligible": "⚪",
        "Unknown": "❓"
    }
    
    sev_counts = df_vulns["severity"].value_counts()
    
    for sev in severity_order:
        if sev in sev_counts.index:
            count = sev_counts[sev]
            pct = count / len(df_vulns) * 100
            icon = severity_colors.get(sev, "")
            bar = "█" * max(1, int(pct / 2))
            print(f"  {icon} {sev:12} │ {bar} {count} ({pct:.1f}%)")
    
    print(f"\n  Total: {len(df_vulns)} vulnerabilidades")


🎯 Distribución por severidad:

  🟠 High         │ ███ 1 (6.7%)
  🟡 Medium       │ ████████████████████████████████████████ 12 (80.0%)
  🟢 Low          │ ██████ 2 (13.3%)

  Total: 15 vulnerabilidades


### 8.2 Vulnerabilidades por repositorio y severidad

In [10]:
if not df_vulns.empty:
    print("\n📊 Vulnerabilidades por repositorio y severidad:\n")
    
    pivot = df_vulns.pivot_table(
        index="repo",
        columns="severity",
        values="vuln_id",
        aggfunc="count",
        fill_value=0
    )
    
    # Reordenar columnas
    cols = [c for c in severity_order if c in pivot.columns]
    pivot = pivot[cols]
    pivot["TOTAL"] = pivot.sum(axis=1)
    pivot = pivot.sort_values("TOTAL", ascending=False)
    
    display(pivot)


📊 Vulnerabilidades por repositorio y severidad:



severity,High,Medium,Low,TOTAL
repo,,,,
flask,1,12,2,15


### 8.3 Paquetes más vulnerables

In [11]:
if not df_vulns.empty:
    print("\n📦 Paquetes con más vulnerabilidades:\n")
    
    pkg_vulns = df_vulns.groupby(["package", "version"]).agg(
        total_vulns=("vuln_id", "count"),
        critical=("severity", lambda x: (x == "Critical").sum()),
        high=("severity", lambda x: (x == "High").sum()),
        medium=("severity", lambda x: (x == "Medium").sum()),
        low=("severity", lambda x: (x == "Low").sum()),
    ).reset_index().sort_values("total_vulns", ascending=False)
    
    display(pkg_vulns)
    
    # Gráfico ASCII
    print("\n📊 Gráfico de vulnerabilidades por paquete:\n")
    max_v = pkg_vulns["total_vulns"].max()
    for _, row in pkg_vulns.iterrows():
        name = f"{row['package']}@{row['version']}"
        bar_len = int(row['total_vulns'] / max_v * 35)
        bar = "🔴" * int(row['critical']) + "🟠" * int(row['high']) + "🟡" * int(row['medium']) + "🟢" * int(row['low'])
        print(f"  {name:30} │ {bar} {int(row['total_vulns'])}")


📦 Paquetes con más vulnerabilidades:



,package,version,total_vulns,critical,high,medium,low
4,werkzeug,2.3.3,7,0,1,6,0
2,jinja2,3.1.2,5,0,0,5,0
0,cryptography,46.0.6,1,0,0,1,0
1,flask,2.3.2,1,0,0,0,1
3,uv,0.11.3,1,0,0,0,1



📊 Gráfico de vulnerabilidades por paquete:

  werkzeug@2.3.3                 │ 🟠🟡🟡🟡🟡🟡🟡 7
  jinja2@3.1.2                   │ 🟡🟡🟡🟡🟡 5
  cryptography@46.0.6            │ 🟡 1
  flask@2.3.2                    │ 🟢 1
  uv@0.11.3                      │ 🟢 1


### 8.4 Estado de correcciones disponibles

In [12]:
if not df_vulns.empty:
    print("\n🔧 Estado de correcciones disponibles:\n")
    
    fix_counts = df_vulns["fix_state"].value_counts()
    
    fix_icons = {
        "fixed": "✅",
        "not-fixed": "❌",
        "wont-fix": "🚫",
        "unknown": "❓",
        "N/A": "❓"
    }
    
    for state, count in fix_counts.items():
        pct = count / len(df_vulns) * 100
        icon = fix_icons.get(state, "")
        print(f"  {icon} {state:15} │ {count:4} ({pct:.1f}%)")
    
    # Vulnerabilidades críticas/altas sin fix
    critical_no_fix = df_vulns[
        (df_vulns["severity"].isin(["Critical", "High"])) &
        (df_vulns["fix_state"] != "fixed")
    ]
    
    if not critical_no_fix.empty:
        print(f"\n  ⚠️  Vulnerabilidades Critical/High SIN fix disponible: {len(critical_no_fix)}")
        display(critical_no_fix[["repo", "vuln_id", "severity", "package", "version"]])
    else:
        print(f"\n  ✅ Todas las vulnerabilidades Critical/High tienen fix disponible")
    
    # Detalle de correcciones disponibles
    fixed_vulns = df_vulns[df_vulns["fix_state"] == "fixed"]
    if not fixed_vulns.empty:
        print(f"\n📋 Versiones de corrección recomendadas:\n")
        fix_details = fixed_vulns[["package", "version", "fix_versions", "severity"]].drop_duplicates()
        fix_details = fix_details.sort_values("severity")
        display(fix_details)


🔧 Estado de correcciones disponibles:

  ✅ fixed           │   15 (100.0%)

  ✅ Todas las vulnerabilidades Critical/High tienen fix disponible

📋 Versiones de corrección recomendadas:



,package,version,fix_versions,severity
0,werkzeug,2.3.3,3.0.3,High
13,flask,2.3.2,3.1.3,Low
14,uv,0.11.3,0.11.6,Low
1,werkzeug,2.3.3,3.0.6,Medium
3,jinja2,3.1.2,3.1.4,Medium
4,werkzeug,2.3.3,2.3.8,Medium
5,jinja2,3.1.2,3.1.5,Medium
7,jinja2,3.1.2,3.1.6,Medium
8,jinja2,3.1.2,3.1.3,Medium
9,cryptography,46.0.6,46.0.7,Medium


---

## 9. 🔍 Análisis de Código Fuente (CodeQL)

Resultados del análisis estático del código fuente.

In [13]:
# Cargar resultados de CodeQL
all_codeql = []

for codeql_file in codeql_files:
    repo_name = codeql_file.stem.replace("-codeql", "")
    
    with open(codeql_file) as f:
        data = json.load(f)
    
    # Soportar formato nuevo (findings) y formato SARIF directo
    if "findings" in data:
        # Formato generado por nuestro conversor SARIF→JSON
        for finding in data["findings"]:
            file_path = "N/A"
            line = 0
            if finding.get("locations"):
                loc = finding["locations"][0]
                file_path = loc.get("file", "N/A")
                line = loc.get("startLine", 0)
            
            all_codeql.append({
                "repo": repo_name,
                "rule_id": finding.get("rule_id", "N/A"),
                "name": finding.get("name", "N/A"),
                "level": finding.get("severity", "warning"),
                "message": finding.get("description", "N/A")[:120],
                "file": file_path,
                "line": line,
            })
    else:
        # Formato SARIF directo
        results = data if isinstance(data, list) else data.get("runs", [{}])[0].get("results", [])
        for result in results:
            if isinstance(result, dict):
                rule_id = result.get("ruleId", result.get("rule", {}).get("id", "N/A"))
                msg = result.get("message", {})
                message = msg.get("text", str(msg)) if isinstance(msg, dict) else str(msg)
                level = result.get("level", "warning")
                
                locations = result.get("locations", [{}])
                file_path = "N/A"
                line = 0
                if locations:
                    phys = locations[0].get("physicalLocation", {})
                    file_path = phys.get("artifactLocation", {}).get("uri", "N/A")
                    line = phys.get("region", {}).get("startLine", 0)
                
                all_codeql.append({
                    "repo": repo_name,
                    "rule_id": rule_id,
                    "name": rule_id,
                    "level": level,
                    "message": message[:120],
                    "file": file_path,
                    "line": line,
                })

df_codeql = pd.DataFrame(all_codeql)

if not df_codeql.empty:
    print(f"\n🔍 Total de hallazgos CodeQL: {len(df_codeql)}")
    print(f"📁 Repositorios analizados: {df_codeql['repo'].nunique()}")
    print(f"📋 Reglas activadas: {df_codeql['rule_id'].nunique()}")
    print(f"\n--- Primeros 15 hallazgos ---")
    display(df_codeql.head(15))
else:
    print("ℹ️  No se encontraron resultados de CodeQL.")
    print("   Esto es normal si no se ejecutó el paso de CodeQL.")


🔍 Total de hallazgos CodeQL: 68
📁 Repositorios analizados: 1
📋 Reglas activadas: 17

--- Primeros 15 hallazgos ---


,repo,rule_id,name,level,message,file,line
0,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/test_async.py,27
1,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/test_async.py,48
2,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/test_async.py,63
3,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/test_basic.py,1482
4,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/test_blueprints.py,278
5,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/test_helpers.py,251
6,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/test_testing.py,126
7,flask,py/reflective-xss,Reflected server-side cross-site scripting,error,Cross-site scripting vulnerability due to a [u...,tests/type_check/typing_route.py,72
8,flask,py/file-not-closed,File is not always closed,warning,File is opened but is not closed.,tests/test_helpers.py,368
9,flask,py/unused-loop-variable,Suspicious unused loop iteration variable,error,For loop variable '_trigger' is not used in th...,tests/test_basic.py,1120


### 9.1 Hallazgos por nivel de severidad

In [14]:
if not df_codeql.empty:
    print("\n📊 Hallazgos por nivel de severidad:\n")
    
    level_icons = {"error": "🔴", "warning": "🟡", "note": "📝"}
    level_counts = df_codeql["level"].value_counts()
    
    for level, count in level_counts.items():
        pct = count / len(df_codeql) * 100
        icon = level_icons.get(level, "❓")
        bar = "█" * max(1, int(pct / 2))
        print(f"  {icon} {level:15} │ {bar} {count} ({pct:.1f}%)")
    
    print(f"\n  Total: {len(df_codeql)} hallazgos")


📊 Hallazgos por nivel de severidad:

  📝 note            │ ███████████████████████████ 37 (54.4%)
  🔴 error           │ ███████████████████ 26 (38.2%)
  🟡 warning         │ ███ 5 (7.4%)

  Total: 68 hallazgos


### 9.2 Top reglas más frecuentes

In [15]:
if not df_codeql.empty:
    print("\n📋 Top 15 reglas más frecuentes:\n")
    
    # Usar 'name' si está disponible, sino 'rule_id'
    name_col = "name" if "name" in df_codeql.columns else "rule_id"
    rule_counts = df_codeql.groupby([name_col, "level"]).size().reset_index(name="count")
    rule_counts = rule_counts.sort_values("count", ascending=False).head(15)
    
    max_count = rule_counts["count"].max()
    for _, row in rule_counts.iterrows():
        icon = level_icons.get(row["level"], "❓")
        bar_len = int(row["count"] / max_count * 25)
        bar = "█" * max(1, bar_len)
        print(f"  {icon} {row[name_col]:50} │ {bar} {row['count']}")


📋 Top 15 reglas más frecuentes:

  📝 Statement has no effect                            │ █████████████████████████ 28
  🔴 An assert statement has a side-effect              │ ████████ 10
  🔴 Reflected server-side cross-site scripting         │ ███████ 8
  📝 Module is imported with 'import' and 'import from' │ ███ 4
  🔴 Unused exception object                            │ ██ 3
  📝 Unused import                                      │ █ 2
  🟡 Unreachable code                                   │ █ 2
  🔴 Missing call to superclass `__init__` during object initialization │ █ 2
  📝 Empty except                                       │ █ 1
  🔴 Potentially uninitialized local variable           │ █ 1
  📝 Module imports itself                              │ █ 1
  🟡 File is not always closed                          │ █ 1
  🔴 Illegal raise                                      │ █ 1
  🔴 Suspicious unused loop iteration variable          │ █ 1
  📝 Unused global variable                            

### 9.3 Archivos más afectados

In [16]:
if not df_codeql.empty:
    print("\n📂 Top 10 archivos con más hallazgos:\n")
    
    file_counts = df_codeql["file"].value_counts().head(10)
    
    max_count = file_counts.max()
    for file_path, count in file_counts.items():
        bar_len = int(count / max_count * 25)
        bar = "█" * max(1, bar_len)
        # Mostrar solo el nombre del archivo, no la ruta completa
        short_path = file_path if len(file_path) < 50 else "..." + file_path[-47:]
        print(f"  {short_path:50} │ {bar} {count}")


📂 Top 10 archivos con más hallazgos:

  src/flask/globals.py                               │ █████████████████████████ 8
  tests/test_blueprints.py                           │ █████████████████████ 7
  src/flask/sansio/app.py                            │ █████████████████████ 7
  src/flask/sansio/blueprints.py                     │ ██████████████████ 6
  tests/type_check/typing_app_decorators.py          │ ████████████ 4
  tests/test_async.py                                │ █████████ 3
  tests/test_basic.py                                │ █████████ 3
  tests/test_helpers.py                              │ █████████ 3
  src/flask/cli.py                                   │ █████████ 3
  src/flask/config.py                                │ █████████ 3


### 9.4 Hallazgos de seguridad críticos (CodeQL)

Filtramos solo los hallazgos nivel **error** que representan vulnerabilidades de seguridad reales.

In [17]:
if not df_codeql.empty:
    errors = df_codeql[df_codeql["level"] == "error"]
    
    if not errors.empty:
        print(f"\n🔴 Hallazgos nivel ERROR (potenciales vulnerabilidades): {len(errors)}\n")
        display(errors[["repo", "name", "message", "file", "line"]].reset_index(drop=True))
    else:
        print("\n✅ No se encontraron hallazgos nivel error (excelente!)")


🔴 Hallazgos nivel ERROR (potenciales vulnerabilidades): 26



,repo,name,message,file,line
0,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/test_async.py,27
1,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/test_async.py,48
2,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/test_async.py,63
3,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/test_basic.py,1482
4,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/test_blueprints.py,278
5,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/test_helpers.py,251
6,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/test_testing.py,126
7,flask,Reflected server-side cross-site scripting,Cross-site scripting vulnerability due to a [u...,tests/type_check/typing_route.py,72
8,flask,Suspicious unused loop iteration variable,For loop variable '_trigger' is not used in th...,tests/test_basic.py,1120
9,flask,Potentially uninitialized local variable,Local variable 'error' may be used before it i...,tests/test_basic.py,1384


---

## 10. 📊 Resumen Ejecutivo

Consolidación de todos los hallazgos en métricas clave.

In [18]:
print("\n" + "═" * 65)
print("        📊 RESUMEN EJECUTIVO DE ANÁLISIS DE SEGURIDAD")
print("═" * 65)
print(f"  Fecha del análisis:      {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Repositorios analizados: {len(sbom_files)}")
repos_names = [f.stem.replace('-sbom', '') for f in sbom_files]
for r in repos_names:
    print(f"    → {r}")
print()

# --- Dependencias ---
print("  ─── DEPENDENCIAS (SBOM) ──────────────────────────")
if not df_deps.empty:
    print(f"  Total de dependencias:       {len(df_deps)}")
    print(f"  Paquetes únicos:             {df_deps['name'].nunique()}")
    print(f"  Tipos de paquete:            {df_deps['type'].nunique()}")
    print(f"  Promedio por repositorio:    {len(df_deps) / max(df_deps['repo'].nunique(), 1):.0f}")
else:
    print("  Sin datos de SBOM")

print()

# --- Vulnerabilidades ---
print("  ─── VULNERABILIDADES (Grype) ─────────────────────")
if not df_vulns.empty:
    print(f"  Total de vulnerabilidades:   {len(df_vulns)}")
    for sev in ["Critical", "High", "Medium", "Low"]:
        count = len(df_vulns[df_vulns['severity'] == sev])
        icon = severity_colors.get(sev, '')
        pad = ' ' * (12 - len(sev))
        print(f"    {icon} {sev}:{pad}  {count}")
    
    fixed = len(df_vulns[df_vulns['fix_state'] == 'fixed'])
    print(f"  Con fix disponible:          {fixed} ({fixed/len(df_vulns)*100:.1f}%)")
    print(f"  Paquetes afectados:          {df_vulns['package'].nunique()}")
else:
    print("  Sin datos de Grype")

print()

# --- CodeQL ---
print("  ─── ANÁLISIS ESTÁTICO (CodeQL) ───────────────────")
if not df_codeql.empty:
    print(f"  Total de hallazgos:          {len(df_codeql)}")
    for level in ["error", "warning", "note"]:
        count = len(df_codeql[df_codeql['level'] == level])
        icon = level_icons.get(level, '')
        pad = ' ' * (12 - len(level))
        print(f"    {icon} {level}:{pad}  {count}")
    print(f"  Reglas únicas activadas:     {df_codeql['rule_id'].nunique()}")
    print(f"  Archivos afectados:          {df_codeql['file'].nunique()}")
else:
    print("  Sin datos de CodeQL")

print()

# --- Métricas de riesgo ---
print("  ─── MÉTRICAS DE RIESGO ───────────────────────────")
if not df_vulns.empty and not df_deps.empty:
    ratio = len(df_vulns) / max(len(df_deps), 1)
    critical_high = len(df_vulns[df_vulns['severity'].isin(['Critical', 'High'])])
    print(f"  Ratio vulns/dependencias:    {ratio:.4f} ({ratio*100:.2f}%)")
    print(f"  Vulns Critical+High:         {critical_high}")
    
    if not df_codeql.empty:
        xss_count = len(df_codeql[df_codeql['name'].str.contains('cross-site', case=False, na=False)])
        if xss_count > 0:
            print(f"  Posibles XSS en código:      {xss_count}")
    
    if critical_high == 0:
        print(f"  Estado general:              🟢 BAJO RIESGO")
    elif critical_high <= 5:
        print(f"  Estado general:              🟡 RIESGO MODERADO")
    else:
        print(f"  Estado general:              🔴 ALTO RIESGO")

print("\n" + "═" * 65)


═════════════════════════════════════════════════════════════════
        📊 RESUMEN EJECUTIVO DE ANÁLISIS DE SEGURIDAD
═════════════════════════════════════════════════════════════════
  Fecha del análisis:      2026-04-14 02:13
  Repositorios analizados: 1
    → flask

  ─── DEPENDENCIAS (SBOM) ──────────────────────────
  Total de dependencias:       121
  Paquetes únicos:             104
  Tipos de paquete:            2
  Promedio por repositorio:    121

  ─── VULNERABILIDADES (Grype) ─────────────────────
  Total de vulnerabilidades:   15
    🔴 Critical:      0
    🟠 High:          1
    🟡 Medium:        12
    🟢 Low:           2
  Con fix disponible:          15 (100.0%)
  Paquetes afectados:          5

  ─── ANÁLISIS ESTÁTICO (CodeQL) ───────────────────
  Total de hallazgos:          68
    🔴 error:         26
    🟡 warning:       5
    📝 note:          37
  Reglas únicas activadas:     17
  Archivos afectados:          25

  ─── MÉTRICAS DE RIESGO ───────────────────────────

---

## 11. 💾 Exportar Datos para Reportes

Exportamos los DataFrames a CSV para uso externo (Excel, Google Sheets, etc.).

In [19]:
export_dir = RESULTS_DIR / "exports"
export_dir.mkdir(exist_ok=True)

print("📂 Exportando datos a CSV...\n")

if not df_deps.empty:
    path = export_dir / "dependencias.csv"
    df_deps.to_csv(path, index=False)
    print(f"  ✅ Dependencias     → {path} ({len(df_deps)} filas)")

if not df_vulns.empty:
    path = export_dir / "vulnerabilidades.csv"
    df_vulns.to_csv(path, index=False)
    print(f"  ✅ Vulnerabilidades → {path} ({len(df_vulns)} filas)")

if not df_codeql.empty:
    path = export_dir / "codeql_hallazgos.csv"
    df_codeql.to_csv(path, index=False)
    print(f"  ✅ CodeQL           → {path} ({len(df_codeql)} filas)")

# Exportar resumen ejecutivo
summary_data = {
    "fecha_analisis": datetime.now().isoformat(),
    "repositorios": repos_names,
    "dependencias": {
        "total": len(df_deps) if not df_deps.empty else 0,
        "paquetes_unicos": int(df_deps['name'].nunique()) if not df_deps.empty else 0,
    },
    "vulnerabilidades": {
        "total": len(df_vulns) if not df_vulns.empty else 0,
        "por_severidad": df_vulns["severity"].value_counts().to_dict() if not df_vulns.empty else {},
        "con_fix": int((df_vulns["fix_state"] == "fixed").sum()) if not df_vulns.empty else 0,
    },
    "codeql": {
        "total_hallazgos": len(df_codeql) if not df_codeql.empty else 0,
        "por_nivel": df_codeql["level"].value_counts().to_dict() if not df_codeql.empty else {},
    }
}

path = export_dir / "resumen_ejecutivo.json"
with open(path, "w") as f:
    json.dump(summary_data, f, indent=2, default=str)
print(f"  ✅ Resumen          → {path}")

print(f"\n📂 Todos los exports en: {export_dir}")

📂 Exportando datos a CSV...

  ✅ Dependencias     → /workspaces/sbom-vuln-analysis/data/results/exports/dependencias.csv (121 filas)
  ✅ Vulnerabilidades → /workspaces/sbom-vuln-analysis/data/results/exports/vulnerabilidades.csv (15 filas)
  ✅ CodeQL           → /workspaces/sbom-vuln-analysis/data/results/exports/codeql_hallazgos.csv (68 filas)
  ✅ Resumen          → /workspaces/sbom-vuln-analysis/data/results/exports/resumen_ejecutivo.json

📂 Todos los exports en: /workspaces/sbom-vuln-analysis/data/results/exports


---

## 📝 Conclusiones


1. **Dependencias:** ...
2. **Vulnerabilidades encontradas:** ...
3. **Severidad predominante:** ...
4. **Paquetes más afectados:** ...
5. **Disponibilidad de correcciones:** ...
6. **Hallazgos de análisis estático:** ...
7. **Recomendaciones:** ...

---
**Curso de Ciberseguridad (ICC610) - 2026**